In [5]:
import os, json
from markdown_it import MarkdownIt
from mdit_py_plugins.front_matter import front_matter_plugin
from mdit_py_plugins.footnote import footnote_plugin
from markdown_it.tree import SyntaxTreeNode

In [3]:
md = (
    MarkdownIt('commonmark', {'breaks':True,'html':True})
    .use(front_matter_plugin)
    .use(footnote_plugin)
    .enable('table')
)
f_name = '3.md'
with open(f_name,"r") as f:
    data = f.read()
tokens = md.parse(data)
html_text = md.render(data)
tokens

[Token(type='front_matter', tag='', nesting=0, attrs={}, map=[0, 8], level=0, children=None, content='title: Marginswap\nsponsor: Marginswap\nslug: 2021-04-marginswap\ndate: 2021-05-03\nfindings: https://github.com/code-423n4/2021-04-marginswap-findings/issues\ncontest: 3', markup='---', info='', meta={}, block=True, hidden=True),
 Token(type='heading_open', tag='h1', nesting=1, attrs={}, map=[10, 11], level=0, children=None, content='', markup='#', info='', meta={}, block=True, hidden=False),
 Token(type='inline', tag='', nesting=0, attrs={}, map=[10, 11], level=1, children=[Token(type='text', tag='', nesting=0, attrs={}, map=None, level=0, children=None, content='Overview', markup='', info='', meta={}, block=False, hidden=False)], content='Overview', markup='', info='', meta={}, block=True, hidden=False),
 Token(type='heading_close', tag='h1', nesting=-1, attrs={}, map=None, level=0, children=None, content='', markup='#', info='', meta={}, block=True, hidden=False),
 Token(type='head

In [9]:
root = SyntaxTreeNode(tokens)
def walk(node, depth=0):
    print("  " * depth, node.type)

    if hasattr(node, "children") and node.children:
        for child in node.children:
            walk(child, depth + 1)
walk(root)

 root
   front_matter
   heading
     inline
       text
   heading
     inline
       text
   paragraph
     inline
       text
   paragraph
     inline
       text
   paragraph
     inline
       text
   heading
     inline
       text
   paragraph
     inline
       text
   bullet_list
     list_item
       paragraph
         inline
           link
             text
     list_item
       paragraph
         inline
           link
             text
     list_item
       paragraph
         inline
           link
             text
     list_item
       paragraph
         inline
           link
             text
     list_item
       paragraph
         inline
           link
             text
   paragraph
     inline
       text
       link
         text
       text
   paragraph
     inline
       text
       link
         text
       text
   heading
     inline
       text
   paragraph
     inline
       text
   paragraph
     inline
       text
   paragraph
     inline
       text
   h

In [ ]:
def extract_text_sessions(node):
    sessions = []
    current = {
        "title": None,
        "content": []
    }
    for child in node.children:
        if child.type == "heading":
            if current["title"] or current["content"]:
                sessions.append(current)
            title = child.children[0].content

            current = {
                "title": title,
                "content": []
            }
        elif child.type == "paragraph":
            if child.children:
                text = "".join(
                    c.content for c in child.children
                    if hasattr(c, "content")
                )
                current["content"].append(text)
    if current["title"] or current["content"]:
        sessions.append(current)

    return sessions
sessions = extract_text_sessions(root)

In [57]:
json_sessions = json.dumps(sessions,indent=4)
os.makedirs('outputs/',exist_ok=True)
fp = 'outputs/json_sessions.json'
with open(fp,"w") as f:
    f.write(json_sessions)

Ast will be a list of dicts (objects of title and content)

In [58]:
ast = json.loads(json_sessions)
ast

[{'title': 'Overview', 'content': []},
 {'title': 'About C4',
  'content': ['Code 432n4 (C4) is an open organization that consists of security researchers, auditors, developers, and individuals with domain expertise in the area of smart contracts.',
   'A C4 code contest is an event in which community participants, referred to as Wardens, review, audit, or analyze smart contract logic in exchange for a bounty provided by sponsoring projects.',
   'During the code contest outlined in this document, C4 conducted an analysis of Marginswap’s smart contract system written in Solidity. The code contest took place between April 2 and April 7, 2021.']},
 {'title': 'Wardens',
  'content': ['5 Wardens contributed reports to the Marginswap code contest:',
   'This contest was judged by [Zak Cole](https://twitter.com/0xzak).',
   'Final report assembled by [sockdrawermoney](https://twitter.com/sockdrawermoney).']},
 {'title': 'Summary',
  'content': ['The C4 analysis yielded an aggregated total of

In [76]:
import re

severity_titles = {
    "High Risk Findings": "high",
    "Medium Risk Findings": "medium",
    "Low Risk Findings": "low",
    "Non-Critical Findings": "non_critical",
    "Gas Optimizations": "gas"
}

FINDING_RE = re.compile(r"\[(H|M|L|N|G)-\d+\]")
FINDING_TITLE_RE = re.compile(
    r"\[\[(?P<id>[A-Z]-\d+)\]\s*(?P<title>.*?)\]"
)
def get_findings_from_sessions(sessions):
    curr_severity = None
    findings = []
    for session in sessions:
        title = session["title"]
        #print(title)
        if title in severity_titles.keys():
            curr_severity = severity_titles[title]
        if FINDING_RE.search(title):
            #print(session["content"])
            match = FINDING_TITLE_RE.search(title)
            if match:
                finding_id = match.group("id")
                finding_title = match.group("title")
                finding = {
                    "severity" : curr_severity,
                    "id" : finding_id,
                    "title" : finding_title,
                    "content" : session["content"]
                }
                findings.append(finding)
    return findings
findings = get_findings_from_sessions(sessions=sessions)
findings
    #print(curr_severity)


[{'severity': 'high',
  'id': 'H-01',
  'title': 'Re-entrancy bug allows inflating balance',
  'content': ['One can call the `MarginRouter.crossSwapExactTokensForTokens` function first with a fake contract disguised as a token pair:\n`crossSwapExactTokensForTokens(0.0001 WETH, 0, [ATTACKER_CONTRACT], [WETH, WBTC])`. When the amounts are computed by the `amounts = UniswapStyleLib.getAmountsOut(amountIn - fees, pairs, tokens);` call, the attacker contract returns fake reserves that yield 1 WBTC for the tiny input. The resulting amount is credited through `registerTrade`. Afterwards, `_swapExactT4T([0.0001 WETH, 1 WBTC], 0, [ATTACKER_CONTRACT], [WETH, WBTC])` is called with the fake pair and token amounts. At some point `_swap` is called, the starting balance is stored in `startingBalance`, and the attacker contract call allows a re-entrancy:',
   'From the ATTACKER_CONTRACT we re-enter the `MarginRouter.crossSwapExactTokensForTokens(30 WETH, 0, WETH_WBTC_PAIR, [WETH, WBTC])` function wit

In [80]:
from openai import OpenAI
import openai
FUNCTION_RE = re.compile(
    r"`([A-Za-z0-9_]+\.[A-Za-z0-9_]+)`"
)
CONTRACT_RE = re.compile(
    r"`([A-Z][A-Za-z0-9_]+)\.sol`"
)
URL_RE = re.compile(r"https?://[^\s)]+")
SYSTEM_PROMPT = """
You are a smart contract security analyst.

Return ONLY valid JSON.
"""

USER_PROMPT = """
Extract structured vulnerability information.

Schema:
{
  "vulnerability_type": "",
  "affected_functions": [],
  "root_cause": "",
  "impact": "",
  "recommendation": ""
}

Finding:
...
"""

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": USER_PROMPT
    }
]
def call_llm_agent_api(messages) -> dict:
        try:
            client = OpenAI(
                base_url="http://localhost:11434/v1",
                api_key="ollama",
                timeout=180.0,
                max_retries=5
            )

            completion = client.chat.completions.create(
                model="qwen3:4b",
                messages=messages,
                #Controls randomness (0 = deterministic)
                temperature=0,
                # Controls token sampling distribution
                top_p=1,
                #Forces valid json output
                response_format={"type": "json_object"}
            )
            #print(completion)

            ans = completion.choices[0].message.content
            return json.loads(ans)

        except openai.APIError as e:
            print(f"OpenAI API returned an API error: {e}")
            raise
        except openai.APIConnectionError as e:
            print(f"Falied to connect to OpenAI API: {e}")
            raise
        except Exception as e:
            print(f"Error at LLM API: {e}")
            raise

In [82]:
for finding in findings:
    finding_prompt = f"""
    Title: {finding["title"]}

    Description:
    {''.join(finding["content"])}
    """
    enriched = call_llm_agent_api(
        "\n".join(finding_prompt)
    )

KeyboardInterrupt: 